# Architecture Overview
`graph.py` acts as the structural spine. It takes the flat, parsed engineering data objects and converts them into an in-memory Directed Multigraph Knowledge Graph using NetworkX.

A Directed Multigraph Knowledge Graph is a knowledge graph where:
- **Directed** → relationships have a specific direction.
- **Multigraph** → two nodes can have multiple different edges/relationships between them.
- **Knowledge graph** → the nodes and relationships represent real-world entities and their semantic relationships.

builds precise structural links across three distinct tiers of project documentation: Contract clauses -> Spec Requirements -> Design Callouts

## Key Concepts
- **NetworkX**: A standard Python library for creating, manipulating, and studying complex network structures, graphs, and node/edge relationships.
- **Directed Graph**: A graph where relationships have direction.
- **Closed-form vs. Unstructured Triples**:
  - **Closed-form (numeric)**: If a contract clause, spec requirement, and drawing callout all contain explicit numerical parameters/tolerances, the triple is classified as numeric and routed to the deterministic engine.
  - **Unstructured (free_text)**: If any element in the triple involves unstructured text or qualitative descriptions, the triple is marked as free_text and routed to the agentic LLM fallback.

## Enumeration Categories
- **triples**: Active triples ready for verification.
- **excluded**: Triples associated with obsolete/superseded drawing revisions.
- **gaps**: Unresolved clauses that lack corresponding specs or drawing callouts.

In [ ]:
"""Directed Multigraph Knowledge Graph using NetworkX"""
"""Builds the NetworkX knowledge graph and enumerates every check (triple)."""

import networkx as nx
from models import Corpus, Triple

norm = lambda s: " ".join(s.lower().split())

In [ ]:
def build_graph(c: Corpus) -> nx.DiGraph:
    G = nx.DiGraph()    # empty directed graph object
    # node creation
    for d in c.documents: G.add_node(d.id, type="document", label=d.title, obj=d)
    for x in c.clauses: G.add_node(x.id, type="clause", label=x.id, obj=x)
    for s in c.spec_reqs: G.add_node(s.id, type="spec", label=s.id, obj=s)
    for r in c.drawing_revs: G.add_node(r.id, type="drawing_rev", label=f"{r.drawing_id} Rev {r.revision}", obj=r)
    for k in c.callouts: G.add_node(k.id, type="callout", label=k.id, obj=k)

    # directed edge creation
    for x in c.clauses:
        if x.spec_ref in G: G.add_edge(x.id, x.spec_ref, rel="REFERENCES")          # dangling refs are reported as ingestion warnings + gaps
        if x.cites_drawing_rev in G: G.add_edge(x.id, x.cites_drawing_rev, rel="CITES")
    # Adds supersession edges
    for r in c.drawing_revs:
        if r.supersedes: G.add_edge(r.id, r.supersedes, rel="SUPERSEDES")   # newer -> older
    return G    # fully populated knowledge graph

In [ ]:
def is_current(G, rev_id: str) -> bool:
    """A revision is current iff nothing SUPERSEDES it."""
    return not any(G.edges[u, rev_id]["rel"] == "SUPERSEDES" for u in G.predecessors(rev_id))

In [ ]:
def enumerate_triples(G):
    """Return (triples to verify, triples excluded as superseded, coverage gaps = clauses with nothing to verify against)."""
    triples, excluded, gaps = [], [], []    # accumulation lists
    
    for n, a in G.nodes(data=True):
        if a["type"] != "clause": continue
        cl = a["obj"]   # stores the underlying Contract Clause object
        if cl.kind == "revision" and cl.cites_drawing_rev not in G:
            gaps.append(dict(clause_id=n, parameter=cl.parameter, reason="cited revision not in drawing register")); continue
        
        mine, old = [], []
        for spec in [m for m in G.successors(n) if G.edges[n, m]["rel"] == "REFERENCES"]:
            for cal in [m for m in G.successors(spec) if G.edges[spec, m]["rel"] == "COVERS"]:
                rev = next((m for m in G.successors(cal)), None)
                if rev is None: continue

                # Deterministic Classifier: Clause, Spec, and Callout
                sp, ca = G.nodes[spec]["obj"], G.nodes[cal]["obj"]
                closed_form = cl.kind == "numeric" and sp.kind == "numeric" and ca.value is not None
                # Triple creation
                t = Triple(id=f"T-{n}-{cal}", kind="numeric" if closed_form else "free_text", clause_id=n, spec_id=spec, drawing_rev_id=rev, callout_id=cal)
                (mine if is_current(G, rev) else old).append(t)
        
        if not mine:
            why = "only superseded drawing callouts" if old else "unknown spec reference" if not cl.spec_ref or cl.spec_ref not in G else "no drawing callout for this requirement"
            gaps.append(dict(clause_id=n, parameter=cl.parameter, reason=why))
    return triples, excluded, gaps